**Not yet run against real data.** Loading/plotting code below was smoke-tested against synthetic fixtures matching the real `.npz` schema (correct shapes, correct `FK_Multi` call signature, layout verified by rendering) but has never seen an actual recording. Re-check every plot visually once real runs exist -- in particular the sign of `force` (see comment below) before trusting `ax.set_ylim(0, None)`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../Kinematics'))
from plot_config import COLORS, FIG_W_SINGLE, FIG_W_DOUBLE, set_font_size
from FK_Multi import FK_Multi
GRID = False
FONT_SIZE = 18
set_font_size(FONT_SIZE)

import numpy as np
import matplotlib.pyplot as plt

## Fig 9 — Exploiting Contact for Button Pressing

`gauss_repel` strength swept across 4 runs (see `ButtonPressing/README.md`); rename each auto-saved run into `outputs/button_pressing/` to match `RUN_FILES` below before running this.

In [ ]:
RUN_FILES = [
    ('no_contact.npz',   'no contact'),
    ('middle_1.npz',     'middle 1'),
    ('middle_2.npz',     'middle 2'),
    ('full_contact.npz', 'full contact'),
]
S_SAMPLES  = [0.0, 1/3, 2/3, 1.0]          # base -> tip, for the posture plot
button_pos = np.array([0.05, 0.0, 0.38])   # GUI_MultiVMC_Config.button_pos

runs = []
for fname, label in RUN_FILES:
    d = np.load(os.path.join('outputs', 'button_pressing', fname), allow_pickle=True)
    t = d['t'] - d['t'][0]
    force_n = d['force'] * 9.80665e-3   # load cell outputs grams -> N (arduino_force.py has no onboard conversion)
    q_motor = d['q_motor']              # (n, 9) [rad]
    runs.append(dict(label=label, t=t, force_n=force_n, q_motor=q_motor))

for r in runs:
    print(f"{r['label']:>14}: {len(r['t'])} samples, {r['t'][-1]:.1f} s, "
          f"force range [{r['force_n'].min():.2f}, {r['force_n'].max():.2f}] N")

In [ ]:
# Force vs. time -- the paper's required quantitative result: contact force
# drops as gauss_repel strength increases from 0 (full contact) to the value
# that fully cancels the approach (no contact).
fig, ax = plt.subplots(figsize=(FIG_W_DOUBLE, FIG_W_DOUBLE * 0.5))
for i, r in enumerate(runs):
    ax.plot(r['t'], r['force_n'], color=COLORS[i], linewidth=1.8, label=r['label'])

ax.set_xlabel('Time [s]')
ax.set_ylabel('Contact force [N]')
ax.grid(GRID)
# arduino_force.py flips sign on publish (msg_shear.data = -force_shear) --
# VERIFY force_n is actually positive-going here before trusting this ylim;
# forcing (0, None) on negative-going data silently produces a blank plot.
ax.set_ylim(0, None)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'button_pressing', 'fig9_force_vs_time.pdf'))
plt.show()

In [ ]:
# Predicted posture (FK backbone) at steady state for each condition -- shows
# spatially how far the tip is pulled back from the button as gauss_repel
# strength increases, alongside the force numbers above.
fig, ax = plt.subplots(figsize=(4.2, 7.5))
for i, r in enumerate(runs):
    n = len(r['q_motor'])
    q_ss = r['q_motor'][int(0.8 * n):].mean(axis=0)   # steady-state window, last 20%
    backbone = np.array([FK_Multi(q_ss, s)[:3] for s in S_SAMPLES])
    ax.plot(backbone[:, 0], backbone[:, 2], '-o', color=COLORS[i],
            linewidth=2.2, markersize=4, label=r['label'], zorder=3)

ax.plot(button_pos[0], button_pos[2], '*', color='k', markersize=15, label='button', zorder=4)
ax.set_xlabel('X [m]')
ax.set_ylabel('Z [m]')
ax.set_aspect('equal')
ax.grid(GRID)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=10, borderaxespad=0.)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'button_pressing', 'fig9_pose.pdf'))
plt.show()

## Fig 10 — Demo with Person

Qualitative photo/video demo (paper's softer "C. Demo" section) -- the primary deliverable is media captured during `person_avoiding.py` runs, not a plot. No figure is generated here; if a supporting force/posture check is ever needed, `outputs/person_avoiding/*.npz` has the same schema as the Fig 9 runs above and can be loaded the same way.